# LeetCode 372 — Super Pow bằng Python

Link đề: https://leetcode.com/problems/super-pow/ · Độ khó: **Medium** · Chủ đề: Math (toán), Divide and Conquer (chia để trị), Modular Arithmetic (số học đồng dư), Euler's Theorem (định lý Euler)

Skeleton LeetCode:

```python
class Solution:
    def superPow(self, a: int, b: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính **`a^b mod 1337`**, trong đó `a` là số nguyên dương (tới `2^31 - 1`) và **`b` là một số cực lớn** được cho dưới dạng **mảng các chữ số** (tới `2000` chữ số), nên **không thể** nhét `b` vào một biến kiểu 32/64 bit trong ngôn ngữ thông thường.
2. Nắm các công cụ: phép **`%` (modulo)** và tính chất `(x * y) % m = ((x % m) * (y % m)) % m`, hàm **`pow(x, e, m)`** (lũy thừa modulo có sẵn), **lũy thừa nhanh (bình phương liên tiếp)**, `divmod`, và `math.gcd`.
3. Đi từ cách **nhân `a` đúng `b` lần** (chỉ dùng được khi `b` nhỏ) → **ghép `b` thành số nguyên rồi lũy thừa nhanh** → **tách từng chữ số: `a^(10x + d) = (a^x)^10 · a^d`** (cách nộp) → **rút gọn số mũ bằng định lý Euler (`φ(1337) = 1140`)** → **tìm chu kỳ của dãy `a^k mod 1337`**.
4. Hiểu mấu chốt: **chỉ cần tính `mod 1337` ở mọi bước**, và một số mũ khổng lồ có thể được **bóc từng chữ số** (mỗi chữ số cần `^10` và `^d`) hoặc **rút gọn** nhờ tính tuần hoàn của lũy thừa modulo.
5. Tránh các lỗi kinh điển: tính `a ** B` trực tiếp (số khổng lồ), **rút gọn số mũ theo `1337`** thay vì `φ(1337)`, áp dụng định lý Euler khi `a` **không nguyên tố cùng nhau** với `1337` mà bỏ sót trường hợp biên, và **nhầm thứ tự** `^10` / `^d` khi bóc chữ số.

## 2. Đề bài & ví dụ

Tính **`a^b mod 1337`**, trong đó `a` là số nguyên dương và **`b` là một số nguyên dương cực lớn** cho dưới dạng **mảng các chữ số** (chữ số cao nhất ở đầu mảng).

**Ví dụ 1**

- Input: `a = 2`, `b = [3]`
- Output: `8`
- Giải thích: `b = 3`, `2^3 = 8`.

**Ví dụ 2**

- Input: `a = 2`, `b = [1,0]`
- Output: `1024`
- Giải thích: `b = 10`, `2^10 = 1024` và `1024 < 1337` nên `1024 mod 1337 = 1024`.

**Ví dụ 3**

- Input: `a = 1`, `b = [4,3,3,8,5,2]`
- Output: `1`
- Giải thích: `1` mũ bao nhiêu cũng là `1`.

**Ví dụ tự dựng (đối chiếu lại đề gốc)**

- `a = 2147483647`, `b = [2,0,0,0]` → một số trong `[0, 1336]` (tự tính bằng code).
- `a = 1337`, `b = [5]` → `0` (vì `1337 mod 1337 = 0`).

**Ràng buộc**

- `1 <= a <= 2^31 - 1`
- `1 <= b.length <= 2000`
- `0 <= b[i] <= 9`
- `b` không có chữ số `0` thừa ở đầu.

Số `b` có thể lên tới `10^2000`: ta không thể lặp `b` lần, cũng không thể tính `a ** b` rồi `% 1337` (kết quả có hàng chục nghìn tỉ chữ số).

In [1]:
from typing import List
import random
import math

MOD = 1337

examples = [
    (2, [3], 8),
    (2, [1, 0], 1024),
    (1, [4, 3, 3, 8, 5, 2], 1),
    (1337, [5], 0),
]
for a, b, expected in examples:
    print(f"a={a}, b={b} -> kỳ vọng {expected}")

a=2, b=[3] -> kỳ vọng 8
a=2, b=[1, 0] -> kỳ vọng 1024
a=1, b=[4, 3, 3, 8, 5, 2] -> kỳ vọng 1
a=1337, b=[5] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Phép `%` và tính chất nhân đồng dư

`x % m` là **số dư** khi chia `x` cho `m`, luôn trong `[0, m - 1]`. Tính chất cực quan trọng: **nhân rồi lấy dư bằng lấy dư rồi nhân rồi lấy dư**:

`(x · y) mod m = ((x mod m) · (y mod m)) mod m`.

Nhờ vậy ta **không bao giờ để số phình to**: sau mỗi phép nhân ta `% 1337` luôn, mọi số trung gian đều `< 1337` (tích hai số nhỏ hơn `1337` nhỏ hơn `~1.8` triệu).

In [2]:
x, y, m = 123456789, 987654321, 1337
print((x * y) % m)
print(((x % m) * (y % m)) % m)         # giống hệt
print(x % m, y % m)

332
332
883 388


### 3.2. Lũy thừa và `pow(x, e, m)`

`x ** e` là lũy thừa. `pow(x, e, m)` tính **`x^e mod m`** rất nhanh, **không** tạo ra số khổng lồ trung gian. Ta sẽ dùng nó làm **oracle** (đáp án đối chiếu) để kiểm tra mọi cách giải. Các luật mũ cần nhớ:

- `a^(x + y) = a^x · a^y`
- `(a^x)^y = a^(x·y)`
- Hệ quả quan trọng cho bài này: **`a^(10x + d) = (a^x)^10 · a^d`**.

In [3]:
print(pow(2, 10, 1337), 2 ** 10 % 1337)           # 1024 1024
print(pow(2, 123, 1337))
a = 2
x, d = 12, 3                                       # 123 = 10 * 12 + 3
print(pow(pow(a, x, 1337), 10, 1337) * pow(a, d, 1337) % 1337)   # cùng đáp án với pow(2, 123, 1337)

1024 1024
1191
1191


### 3.3. Từ mảng chữ số về số nguyên: phương pháp Horner

Mảng `[1, 2, 3]` biểu diễn số `123`. Đọc từ trái sang phải: `n = n * 10 + digit`. `[1,2,3]`: `0 → 1 → 12 → 123`. Python có số nguyên không giới hạn nên làm được cả với 2000 chữ số; tuy vậy ý tưởng "đọc từng chữ số" cũng là chìa khóa để **không cần gom thành số lớn**.

In [4]:
n = 0
for d in [1, 2, 3]:
    n = n * 10 + d
    print("đọc", d, "-> n =", n)

big = 0
for d in [9] * 2000:
    big = big * 10 + d
print("số 2000 chữ số 9 có", len(str(big)), "chữ số")

đọc 1 -> n = 1
đọc 2 -> n = 12
đọc 3 -> n = 123
số 2000 chữ số 9 có 2000 chữ số


### 3.4. Lũy thừa nhanh (bình phương liên tiếp)

Để tính `x^e` mà không nhân `e` lần: viết `e` dưới dạng **nhị phân**. Ví dụ `e = 13 = 1101₂` nên `x^13 = x^8 · x^4 · x^1`. Duyệt các bit của `e` từ thấp lên cao: giữ `x`, `x^2`, `x^4`, ... (mỗi lần **bình phương**), và nhân vào kết quả khi bit hiện tại là `1`. Chỉ cần `~log2(e)` bước.

In [5]:
def fast_pow(x, e, m):
    """x^e mod m bằng bình phương liên tiếp."""
    result = 1
    x %= m
    while e:
        if e & 1:                    # bit thấp của e là 1: nhân x hiện tại vào kết quả
            result = result * x % m
        x = x * x % m                # x -> x^2 -> x^4 -> ...
        e >>= 1                      # bỏ bit thấp
    return result


print(fast_pow(2, 13, 1000), pow(2, 13, 1000))
print(fast_pow(3, 10 ** 18, 1337), pow(3, 10 ** 18, 1337))

192 192
900 900


### 3.5. Phi hàm Euler `φ(m)`, `gcd` và định lý Euler

`gcd(a, m)` là ước chung lớn nhất. **`φ(m)`** là số các số trong `1..m` nguyên tố cùng nhau với `m`. **Định lý Euler:** nếu `gcd(a, m) = 1` thì `a^φ(m) ≡ 1 (mod m)`. Do đó `a^e ≡ a^(e mod φ(m)) (mod m)` khi `gcd(a, m) = 1`.

Với `m = 1337 = 7 × 191` (cả hai đều nguyên tố): `φ(1337) = φ(7) · φ(191) = 6 × 190 = 1140`. Khi `gcd(a, 1337) > 1` (tức `a` chia hết cho `7` hoặc `191`), định lý Euler **không áp dụng nguyên văn** — ta sẽ xử lý ở Cách 4.

In [6]:
def euler_phi(n):
    result, p, m = n, 2, n
    while p * p <= m:
        if m % p == 0:
            while m % p == 0:
                m //= p
            result -= result // p
        p += 1
    if m > 1:
        result -= result // m
    return result


print("1337 = 7 * 191:", 7 * 191, "| φ(1337) =", euler_phi(1337), "| φ(7) * φ(191) =", 6 * 190)
print("gcd(3, 1337) =", math.gcd(3, 1337), "-> 3^1140 mod 1337 =", pow(3, 1140, 1337))
print("gcd(7, 1337) =", math.gcd(7, 1337), "-> 7^1140 mod 1337 =", pow(7, 1140, 1337), "(không phải 1)")

1337 = 7 * 191: 1337 | φ(1337) = 1140 | φ(7) * φ(191) = 1140
gcd(3, 1337) = 1 -> 3^1140 mod 1337 = 1
gcd(7, 1337) = 7 -> 7^1140 mod 1337 = 574 (không phải 1)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính `2^123 mod 1337` mà **không** gom `123`: ta bóc từng chữ số của `b = [1, 2, 3]` từ trái sang phải. Gọi `result` là **`a^(phần số đã đọc)`**.

- Ban đầu chưa đọc gì: số đã đọc là `0`, `result = a^0 = 1`.
- Đọc chữ số `1`: số mới `= 0 · 10 + 1 = 1`. `a^1 = (a^0)^10 · a^1 = result^10 · a^1`.
- Đọc chữ số `2`: số mới `= 1 · 10 + 2 = 12`. `a^12 = (a^1)^10 · a^2`.
- Đọc chữ số `3`: số mới `= 12 · 10 + 3 = 123`. `a^123 = (a^12)^10 · a^3`.

Tổng quát: khi thêm chữ số `d` vào bên phải số hiện tại `x`, ta có `x' = 10x + d`, nên

**`a^x' = (a^x)^10 · a^d`**.

| Đọc | Số hiện tại | `result` mới |
|---|---|---|
| (bắt đầu) | 0 | `a^0 = 1` |
| 1 | 1 | `1^10 · a^1 = a` |
| 2 | 12 | `a^10 · a^2 = a^12` |
| 3 | 123 | `(a^12)^10 · a^3 = a^123` |

Mỗi bước chỉ cần **hai lũy thừa nhỏ** (mũ `10` và mũ `d ≤ 9`), cộng với phép `% 1337` — dù `b` dài tới `2000` chữ số vẫn chỉ `2000` bước.

In [7]:
a, b = 2, [1, 2, 3]
result = 1
for d in b:
    result = pow(result, 10, MOD) * pow(a, d, MOD) % MOD
    print(f"đọc chữ số {d}: result = {result}")
print("đối chiếu pow(2, 123, 1337) =", pow(2, 123, 1337))

đọc chữ số 1: result = 2
đọc chữ số 2: result = 85
đọc chữ số 3: result = 1191
đối chiếu pow(2, 123, 1337) = 1191


## 5. Cách 1 — Brute Force / baseline: nhân `a` đúng `b` lần

**Ý tưởng.** Ghép `b` thành số nguyên `B`, rồi nhân dồn: `result = 1`; lặp `B` lần `result = result * a % 1337`. Đúng định nghĩa, nhưng chỉ dùng được khi `B` **nhỏ**.

In [8]:
def digits_to_int(b):
    n = 0
    for d in b:
        n = n * 10 + d
    return n


def super_pow_naive(a, b):
    B = digits_to_int(b)
    result = 1
    for _ in range(B):                # lặp B lần: chỉ dùng cho B nhỏ
        result = result * a % MOD
    return result


for a, b, expected in examples[:3]:
    if digits_to_int(b) <= 10 ** 6:
        print(super_pow_naive(a, b), "(kỳ vọng", expected, ")")
    else:
        print("bỏ qua: B quá lớn cho vòng lặp", expected)

8 (kỳ vọng 8 )
1024 (kỳ vọng 1024 )
1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** **Time `O(B)`** — với `b` có `2000` chữ số, `B ~ 10^2000`: **không bao giờ chạy xong**. **Space `O(1)`**.

**Nút thắt:** số lần lặp tỉ lệ với **giá trị** `B`, trong khi ta chỉ nên tốn thời gian theo **số chữ số** (hoặc số bit) của `B`. Cần cách tính lũy thừa **nhanh hơn rất nhiều**.

## 6. Ý tưởng thuật toán / bước đột phá

**Hai bước đột phá, dùng được độc lập:**

1. **Chia đôi số mũ (lũy thừa nhanh):** `x^e = (x^(e/2))^2` (và nhân thêm `x` nếu `e` lẻ). Số mũ giảm một nửa mỗi bước ⇒ chỉ `log2(B) ~ 6644` bước cho `B ~ 10^2000`.
2. **Bóc từng chữ số của số mũ:** `a^(10x + d) = (a^x)^10 · a^d`. Chỉ cần `2000` bước (mỗi bước hai lũy thừa nhỏ) và **không cần gom `b` thành số lớn**.

Ngoài ra còn hai hướng dựa vào **tính tuần hoàn**: dãy `a^0, a^1, a^2, ... (mod 1337)` **lặp lại** vì chỉ có `1337` giá trị có thể. Nếu biết chu kỳ `λ` thì `a^B` chỉ phụ thuộc `B mod λ` — và `B mod λ` tính được **từng chữ số một** bằng `r = (r · 10 + d) % λ`. Định lý Euler cho ta một chu kỳ "đảm bảo" (`φ(1337) = 1140`) khi `gcd(a, 1337) = 1`; hoặc ta tự **dò chu kỳ** ngay bằng mô phỏng.

In [9]:
# Dãy a^k mod 1337 thật sự tuần hoàn: xem một vài giá trị
for a in (2, 7, 1337):
    seq = [pow(a, k, MOD) for k in range(12)]
    print(f"a={a:4d}: {seq}")

a=   2: [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 711]
a=   7: [1, 7, 49, 343, 1064, 763, 1330, 1288, 994, 273, 574, 7]
a=1337: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 7. Các cách tiếp theo

Ngoài Cách 1 (nhân `b` lần), ta có thêm **4 cách khác nhau về ý tưởng**: Cách 2 (ghép `b` thành số rồi lũy thừa nhanh), Cách 3 (bóc từng chữ số — cách nộp), Cách 4 (rút gọn số mũ bằng định lý Euler), Cách 5 (dò chu kỳ của dãy lũy thừa).

### Cách 2 — Ghép `b` thành số nguyên, rồi lũy thừa nhanh

**Giải thích từng đoạn.**

1. Ghép mảng thành số nguyên `B` bằng Horner (Python hỗ trợ số nguyên lớn tùy ý, `2000` chữ số hoàn toàn ổn).
2. Gọi `fast_pow(a, B, 1337)` — bình phương liên tiếp: mỗi bước nhìn bit thấp của `B`, bình phương `x`, rồi dịch `B` sang phải một bit. Khoảng `6644` bit ⇒ `~6644` bước.
3. Mọi phép nhân đều kèm `% 1337`.

Điểm cần lưu ý: cách này **phụ thuộc** việc ngôn ngữ có số nguyên lớn (Python/Java `BigInteger`). Trong C++ bạn không ghép được `B` — lúc đó dùng Cách 3.

In [10]:
def super_pow_fastpow(a, b):
    B = digits_to_int(b)                 # số nguyên lớn của Python
    return fast_pow(a, B, MOD)


for a, b, expected in examples:
    print(super_pow_fastpow(a, b), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
1024 (kỳ vọng 1024 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionFastPow:
    def superPow(self, a: int, b: List[int]) -> int:
        mod = 1337
        e = 0
        for d in b:
            e = e * 10 + d
        result, x = 1, a % mod
        while e:
            if e & 1:
                result = result * x % mod
            x = x * x % mod
            e >>= 1
        return result


print(SolutionFastPow().superPow(2, [1, 0]))     # 1024

1024


**Độ phức tạp Cách 2:** gom `B` tốn `O(n²)` theo kích thước số lớn (với `n = 2000` chữ số là rất nhỏ), lũy thừa nhanh `O(n)` bước nhân modulo nhỏ (`log2(10^n) ~ 3.3 n`, mỗi bước còn dịch bit số lớn cỡ `O(n)` từ). **Time `~O(n²)` thực tế rất nhanh**, **Space `O(n)`** (lưu số `B`).

### Cách 3 — Bóc từng chữ số: `a^(10x + d) = (a^x)^10 · a^d` (cách nộp)

**Giải thích từng đoạn.**

1. `a %= 1337`: mọi tính toán đều trên số nhỏ.
2. `result = 1` tương ứng "đã đọc số 0" (`a^0 = 1`).
3. Với mỗi chữ số `d` của `b` từ trái sang phải: `result = (result^10 mod 1337) · (a^d mod 1337) mod 1337`. Vì `result` đang là `a^x` còn `d` là chữ số mới.
4. Kết thúc: `result = a^B mod 1337`.

Đây là **chia để trị** theo chữ số cuối: viết dưới dạng đệ quy `superPow(a, b) = superPow(a, b[:-1])^10 · a^(b[-1])`. Cả bản lặp và bản đệ quy đều đúng; bản lặp tránh giới hạn đệ quy (`n = 2000` vượt giới hạn mặc định `1000` của Python!).

In [12]:
def super_pow_digits(a, b):
    a %= MOD
    result = 1                                         # a^0
    for d in b:                                        # trái -> phải
        result = fast_pow(result, 10, MOD) * fast_pow(a, d, MOD) % MOD
    return result


for a, b, expected in examples:
    print(super_pow_digits(a, b), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
1024 (kỳ vọng 1024 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [13]:
# Bản đệ quy (chia để trị theo chữ số cuối): chỉ nên dùng khi len(b) nhỏ (giới hạn đệ quy của Python)
def super_pow_recursive(a, b):
    if not b:
        return 1                                       # b rỗng: số mũ 0 -> a^0
    last = b[-1]
    rest = super_pow_recursive(a, b[:-1])              # a^(b không có chữ số cuối)
    return fast_pow(rest, 10, MOD) * fast_pow(a, last, MOD) % MOD


for a, b, expected in examples:
    print(super_pow_recursive(a, b), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
1024 (kỳ vọng 1024 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [14]:
# Code nộp LeetCode cho Cách 3 (bản lặp: an toàn với len(b) = 2000)
class SolutionDigits:
    def superPow(self, a: int, b: List[int]) -> int:
        mod = 1337
        a %= mod
        result = 1
        for d in b:
            result = pow(result, 10, mod) * pow(a, d, mod) % mod
        return result


print(SolutionDigits().superPow(2, [1, 0]), SolutionDigits().superPow(1, [4, 3, 3, 8, 5, 2]))

1024 1


**Độ phức tạp Cách 3:** `n = len(b)` bước, mỗi bước hai lũy thừa với mũ `≤ 10` (hằng số) ⇒ **Time `O(n)`**, **Space `O(1)`** (bản lặp), `O(n)` ngăn xếp (bản đệ quy).

### Cách 4 — Rút gọn số mũ bằng định lý Euler (`φ(1337) = 1140`)

**Giải thích từng đoạn.**

1. Nếu `gcd(a, 1337) = 1`: `a^B ≡ a^(B mod 1140)`. Ta tính `r = B mod 1140` **từng chữ số một**: `r = (r · 10 + d) % 1140` — không cần số lớn.
2. Nếu `gcd(a, 1337) > 1` (tức `7 | a` hoặc `191 | a`): dùng **dạng mở rộng**: với mọi `B >= 1140` (đủ lớn), **`a^B ≡ a^(B mod φ + φ) (mod m)`** (đúng với mọi `a`). Do đó ta lấy số mũ `r + 1140` khi `B >= 1140`, còn khi `B < 1140` thì dùng đúng `B` (vì lúc đó `B mod 1140 = B`).
3. Cuối cùng lũy thừa nhanh với số mũ nhỏ (`< 2280`).

Vì sao dạng mở rộng cần `+ φ`? Ví dụ `a = 7`, `B = 1140`: thực ra `7^1140 mod 1337` **không** bằng `7^0 = 1` (xem mục 13). Cộng thêm `φ` đảm bảo số mũ **đủ lớn** để các thừa số nguyên tố chung với `m` đã bị "hấp thụ" đúng.

In [15]:
PHI = 1140                                   # φ(1337) = φ(7) * φ(191) = 6 * 190


def super_pow_euler(a, b):
    a %= MOD
    r = 0
    for d in b:
        r = (r * 10 + d) % PHI               # r = B mod 1140, tính từng chữ số
    big = len(b) > 4 or digits_to_int(b) >= PHI    # B >= 1140 ?
    e = r + PHI if big else r                # dạng mở rộng: B mod φ + φ khi B đủ lớn
    return fast_pow(a, e, MOD)


for a, b, expected in examples:
    print(super_pow_euler(a, b), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
1024 (kỳ vọng 1024 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionEuler:
    def superPow(self, a: int, b: List[int]) -> int:
        mod, phi = 1337, 1140
        r = 0
        for d in b:
            r = (r * 10 + d) % phi
        big = len(b) > 4 or int("".join(map(str, b))) >= phi
        return pow(a % mod, r + phi if big else r, mod)


print(SolutionEuler().superPow(7, [1, 1, 4, 0]), pow(7, 1140, 1337))     # trường hợp biên B = φ

574 574


**Độ phức tạp Cách 4:** **Time `O(n + log φ)`**, **Space `O(1)`**.

### Cách 5 — Dò chu kỳ của dãy `a^k mod 1337`

**Giải thích từng đoạn.** Không cần biết định lý Euler. Mô phỏng dãy `x_0 = 1, x_1 = a, x_2 = a^2, ... (mod 1337)`: vì chỉ có `1337` giá trị, **sớm muộn sẽ gặp lại một giá trị đã thấy**.

1. Lưu `seen[giá_trị] = k` (lần đầu thấy ở số mũ `k`) và `seq[k]`.
2. Khi gặp lại giá trị `x` đã có ở `μ = seen[x]` tại bước `k`: dãy bắt đầu **lặp** từ `μ` với **chu kỳ `λ = k - μ`**.
3. Cho số mũ `B`: nếu `B < k` thì đáp án là `seq[B]` (chưa vào vòng lặp lại); nếu không thì đáp án là `seq[μ + (B - μ) mod λ]`.
4. `B mod λ` tính từng chữ số: `r = (r · 10 + d) % λ`.

Cách này **thể hiện trực quan** ý "lũy thừa modulo tuần hoàn" và đúng cho **mọi** `a` (kể cả không nguyên tố cùng nhau).

In [17]:
def find_cycle(a, m=MOD):
    """Trả về (seq, mu, lam): seq[k] = a^k mod m, dãy lặp từ vị trí mu với chu kỳ lam."""
    seen, seq = {}, []
    x, k = 1 % m, 0
    while x not in seen:
        seen[x] = k
        seq.append(x)
        x = x * a % m
        k += 1
    mu = seen[x]
    return seq, mu, k - mu


for a in (2, 7, 1337, 3):
    seq, mu, lam = find_cycle(a % MOD)
    print(f"a={a:4d}: độ dài dãy trước khi lặp = {len(seq)}, mu = {mu}, lam = {lam}")

a=   2: độ dài dãy trước khi lặp = 285, mu = 0, lam = 285
a=   7: độ dài dãy trước khi lặp = 11, mu = 1, lam = 10
a=1337: độ dài dãy trước khi lặp = 2, mu = 1, lam = 1
a=   3: độ dài dãy trước khi lặp = 570, mu = 0, lam = 570


In [18]:
def super_pow_cycle(a, b):
    seq, mu, lam = find_cycle(a % MOD)
    k = len(seq)                                   # = mu + lam
    if len(b) <= 4 and digits_to_int(b) < k:       # B nhỏ: chưa vào chu kỳ lặp
        return seq[digits_to_int(b)]
    r = 0
    for d in b:
        r = (r * 10 + d) % lam                     # B mod lam
    return seq[mu + (r - mu) % lam]


for a, b, expected in examples:
    print(super_pow_cycle(a, b), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
1024 (kỳ vọng 1024 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [19]:
# Code nộp LeetCode cho Cách 5
class SolutionCycle:
    def superPow(self, a: int, b: List[int]) -> int:
        mod = 1337
        a %= mod
        seen, seq = {}, []
        x = 1
        while x not in seen:
            seen[x] = len(seq)
            seq.append(x)
            x = x * a % mod
        mu = seen[x]
        lam = len(seq) - mu
        if len(b) <= 4:
            B = int("".join(map(str, b)))
            if B < len(seq):
                return seq[B]
        r = 0
        for d in b:
            r = (r * 10 + d) % lam
        return seq[mu + (r - mu) % lam]


print(SolutionCycle().superPow(2, [1, 0]), SolutionCycle().superPow(7, [1, 1, 4, 0]))

1024 574


**Độ phức tạp Cách 5:** dò chu kỳ tối đa `1337` bước, rồi `O(n)` để tính `B mod λ` ⇒ **Time `O(1337 + n)`**, **Space `O(1337)`**.

## 8. TRACE từng bước

Ta trace **Cách 3** (bóc từng chữ số) trên `a = 2`, `b = [1, 2, 3]`, **Cách 4** (Euler) trên `a = 7`, `b = [1, 1, 4, 0]` (trường hợp biên `B = 1140`, `gcd(7, 1337) = 7`), và **Cách 5** (chu kỳ) trên `a = 3`.

Bảng trace Cách 3 với `a = 2`, `b = [1, 2, 3]` (`mod 1337`):

| Đọc `d` | `result` trước | `result^10 mod 1337` | `a^d` | `result` sau |
|---|---|---|---|---|
| 1 | 1 | 1 | 2 | 2 |
| 2 | 2 | 1024 | 4 | 1024·4 mod 1337 = 85 |
| 3 | 85 | (tính bằng code) | 8 | `2^123 mod 1337` |

In [20]:
def trace_digits(a, b):
    a %= MOD
    result = 1
    print(f"a mod {MOD} = {a}, b = {b}")
    for i, d in enumerate(b):
        before = result
        p10 = pow(result, 10, MOD)
        pd = pow(a, d, MOD)
        result = p10 * pd % MOD
        print(f"bước {i}: đọc d={d}: result trước = {before}, result^10 = {p10}, a^d = {pd} -> result = {p10} * {pd} mod {MOD} = {result}")
    return result


assert trace_digits(2, [1, 2, 3]) == pow(2, 123, MOD)

a mod 1337 = 2, b = [1, 2, 3]
bước 0: đọc d=1: result trước = 1, result^10 = 1, a^d = 2 -> result = 1 * 2 mod 1337 = 2
bước 1: đọc d=2: result trước = 2, result^10 = 1024, a^d = 4 -> result = 1024 * 4 mod 1337 = 85
bước 2: đọc d=3: result trước = 85, result^10 = 316, a^d = 8 -> result = 316 * 8 mod 1337 = 1191


In [21]:
def trace_euler(a, b):
    a %= MOD
    print(f"a mod {MOD} = {a}, gcd(a, {MOD}) = {math.gcd(a, MOD)}, b = {b}")
    r = 0
    for d in b:
        r = (r * 10 + d) % PHI
        print(f"  đọc d={d}: r = (r*10 + d) % {PHI} = {r}")
    big = len(b) > 4 or digits_to_int(b) >= PHI
    e = r + PHI if big else r
    print(f"  B >= {PHI}? {big} -> số mũ dùng = {e}")
    ans = fast_pow(a, e, MOD)
    print(f"  a^{e} mod {MOD} = {ans} | đối chiếu pow(a, B, {MOD}) = {pow(a, digits_to_int(b), MOD)}")
    return ans


assert trace_euler(7, [1, 1, 4, 0]) == pow(7, 1140, MOD)
print()
assert trace_euler(2, [2, 0, 0, 0, 0, 0, 3]) == pow(2, 2000003, MOD)

a mod 1337 = 7, gcd(a, 1337) = 7, b = [1, 1, 4, 0]
  đọc d=1: r = (r*10 + d) % 1140 = 1
  đọc d=1: r = (r*10 + d) % 1140 = 11
  đọc d=4: r = (r*10 + d) % 1140 = 114
  đọc d=0: r = (r*10 + d) % 1140 = 0
  B >= 1140? True -> số mũ dùng = 1140
  a^1140 mod 1337 = 574 | đối chiếu pow(a, B, 1337) = 574

a mod 1337 = 2, gcd(a, 1337) = 1, b = [2, 0, 0, 0, 0, 0, 3]
  đọc d=2: r = (r*10 + d) % 1140 = 2
  đọc d=0: r = (r*10 + d) % 1140 = 20
  đọc d=0: r = (r*10 + d) % 1140 = 200
  đọc d=0: r = (r*10 + d) % 1140 = 860
  đọc d=0: r = (r*10 + d) % 1140 = 620
  đọc d=0: r = (r*10 + d) % 1140 = 500
  đọc d=3: r = (r*10 + d) % 1140 = 443
  B >= 1140? True -> số mũ dùng = 1583
  a^1583 mod 1337 = 1159 | đối chiếu pow(a, B, 1337) = 1159


In [22]:
def trace_cycle(a, b):
    a %= MOD
    seq, mu, lam = find_cycle(a)
    print(f"a = {a}: {len(seq)} giá trị khác nhau trước khi lặp; mu = {mu}, lam = {lam}")
    print("  8 giá trị đầu của dãy:", seq[:8])
    r = 0
    for d in b:
        r = (r * 10 + d) % lam
    print(f"  B mod lam = {r}; chỉ số = mu + (r - mu) mod lam = {mu + (r - mu) % lam}")
    ans = super_pow_cycle(a, b)
    print("  đáp án:", ans, "| đối chiếu:", pow(a, digits_to_int(b), MOD))
    return ans


assert trace_cycle(3, [9, 9, 9, 9, 9]) == pow(3, 99999, MOD)

a = 3: 570 giá trị khác nhau trước khi lặp; mu = 0, lam = 570
  8 giá trị đầu của dãy: [1, 3, 9, 27, 81, 243, 729, 850]
  B mod lam = 249; chỉ số = mu + (r - mu) mod lam = 249
  đáp án: 1014 | đối chiếu: 1014


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (bóc từng chữ số): ngắn, `O(n)` thời gian, `O(1)` bộ nhớ, **không đệ quy** nên an toàn với `n = 2000`, và **không phụ thuộc** số nguyên lớn.

In [23]:
class Solution:
    def superPow(self, a: int, b: List[int]) -> int:
        mod = 1337
        a %= mod                                   # luôn làm việc với số nhỏ
        result = 1                                 # a^0
        for d in b:                                # đọc từng chữ số từ trái sang phải
            result = pow(result, 10, mod) * pow(a, d, mod) % mod   # a^(10x + d) = (a^x)^10 * a^d
        return result


sol = Solution()
for a, b, expected in examples:
    print(a, b, "->", sol.superPow(a, b), "(kỳ vọng", expected, ")")

2 [3] -> 8 (kỳ vọng 8 )
2 [1, 0] -> 1024 (kỳ vọng 1024 )
1 [4, 3, 3, 8, 5, 2] -> 1 (kỳ vọng 1 )
1337 [5] -> 0 (kỳ vọng 0 )


## 10. Vì sao thuật toán đúng

- **Bất biến vòng lặp.** Sau khi đọc `k` chữ số đầu của `b` (tạo thành số nguyên `x_k`), `result ≡ a^(x_k) (mod 1337)`. Ban đầu `k = 0`, `x_0 = 0`, `result = 1 = a^0`.
- **Bước chuyển.** Chữ số tiếp theo `d` cho `x_(k+1) = 10·x_k + d`. Theo luật mũ, `a^(x_(k+1)) = a^(10·x_k + d) = (a^(x_k))^10 · a^d`. Thay `a^(x_k)` bằng `result` (đồng dư theo tính chất nhân đồng dư) được `result^10 · a^d (mod 1337)` — đúng công thức trong code.
- **Kết thúc.** Sau `n` chữ số, `x_n = B` nên `result ≡ a^B (mod 1337)`, và luôn nằm trong `[0, 1336]` do `% mod`.
- **`a %= mod` không đổi đáp án.** `a ≡ a mod 1337`, nên mọi lũy thừa của chúng đồng dư theo `mod 1337`.
- **Cách 2** đúng vì bình phương liên tiếp là phân tích nhị phân của số mũ. **Cách 4** đúng nhờ định lý Euler (với `gcd = 1`) và dạng mở rộng `a^B ≡ a^(B mod φ + φ)` cho `B >= φ` (mọi `a`). **Cách 5** đúng vì một dãy `x_(k+1) = f(x_k)` trên tập hữu hạn là **tuần hoàn từ một lúc nào đó**.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(b)` (tối đa `2000`).

| Cách | Time | Space |
|---|---|---|
| 1. Nhân `B` lần | `O(B)` (không khả thi) | `O(1)` |
| 2. Ghép số + lũy thừa nhanh | `~O(n²)` (số lớn) | `O(n)` |
| 3. Bóc từng chữ số | `O(n)` | `O(1)` |
| 4. Rút gọn bằng định lý Euler | `O(n)` | `O(1)` |
| 5. Dò chu kỳ | `O(1337 + n)` | `O(1337)` |

Với `n = 2000`: Cách 3 chỉ `4000` lũy thừa nhỏ — tức thì. Cách 1 chỉ dùng được cho `B <= ~10^6` (làm oracle phụ cho `B` nhỏ).

## 12. Bộ test

**Oracle độc lập:** `pow(a, B, 1337)` của Python với `B` là số nguyên ghép từ `b` (hàm dựng sẵn bằng thuật toán khác hẳn code của ta). Mọi cách trả về **một số duy nhất** nên so bằng `==`.

In [24]:
sol = Solution()
impls = {
    "naive": super_pow_naive,
    "fastpow": super_pow_fastpow,
    "digits": super_pow_digits,
    "euler": super_pow_euler,
    "cycle": super_pow_cycle,
    "leetcode": sol.superPow,
}


def oracle(a, b):
    return pow(a, digits_to_int(b), MOD)


def check_all(a, b, expected, label, skip=()):
    for key, fn in impls.items():
        if key in skip:
            continue
        actual = fn(a, list(b))
        assert actual == expected, (label, key, a, b, actual, expected)
    assert oracle(a, b) == expected, (label, "oracle")
    print("PASS", label)


for idx, (a, b, expected) in enumerate(examples, 1):
    check_all(a, b, expected, f"ví dụ {idx}", skip=("naive",) if digits_to_int(b) > 10 ** 6 else ())

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3
PASS ví dụ 4


### 12.1. Các trường hợp biên

In [25]:
check_all(1, [1], 1, "a = 1, b = 1")
check_all(2, [1], 2, "b = 1")
check_all(1337, [1], 0, "a = 1337 (≡ 0), b = 1")
check_all(2674, [9, 9], 0, "a = 2 * 1337")
check_all(7, [1, 1, 4, 0], pow(7, 1140, 1337), "B = φ(1337), a = 7 (không nguyên tố cùng nhau)")
check_all(191, [1, 1, 4, 0], pow(191, 1140, 1337), "B = φ, a = 191")
check_all(7 * 191 + 5, [1, 1, 4, 0], pow(5, 1140, 1337), "B = φ, a ≡ 5")
check_all(3, [1, 1, 4, 0], 1, "B = φ, a nguyên tố cùng nhau với 1337 => 1")
check_all(3, [2, 2, 8, 0], 1, "B = 2φ, a nguyên tố cùng nhau => 1")
check_all(7, [2, 2, 8, 0], pow(7, 2280, 1337), "B = 2φ, a = 7")
check_all(2 ** 31 - 1, [9], pow(2 ** 31 - 1, 9, 1337), "a lớn nhất, b một chữ số")
check_all(2 ** 31 - 1, [2, 0, 0, 0], pow(2 ** 31 - 1, 2000, 1337), "a lớn nhất, b = 2000")
check_all(5, [1, 0, 0, 0, 0, 0], pow(5, 10 ** 5, 1337), "b = 100000")

PASS a = 1, b = 1
PASS b = 1
PASS a = 1337 (≡ 0), b = 1
PASS a = 2 * 1337
PASS B = φ(1337), a = 7 (không nguyên tố cùng nhau)
PASS B = φ, a = 191
PASS B = φ, a ≡ 5
PASS B = φ, a nguyên tố cùng nhau với 1337 => 1
PASS B = 2φ, a nguyên tố cùng nhau => 1
PASS B = 2φ, a = 7
PASS a lớn nhất, b một chữ số
PASS a lớn nhất, b = 2000


PASS b = 100000


In [26]:
# b rất dài (2000 chữ số): chỉ chạy các cách nhanh
b_long = [random.Random(372).randint(1, 9)] + [random.Random(i).randint(0, 9) for i in range(1999)]
assert len(b_long) == 2000 and b_long[0] != 0
for a in (2, 7, 1337, 2 ** 31 - 1, 123456789):
    ref = oracle(a, b_long)
    for key in ("fastpow", "digits", "euler", "cycle", "leetcode"):
        assert impls[key](a, list(b_long)) == ref, (key, a)
print("PASS b dài 2000 chữ số (5 giá trị a, 5 hàm)")
assert sol.superPow(9, [9] * 2000) == pow(9, int("9" * 2000), MOD)
print("PASS b = 2000 chữ số 9")

PASS b dài 2000 chữ số (5 giá trị a, 5 hàm)
PASS b = 2000 chữ số 9


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi hay gặp: (a) **rút gọn số mũ theo `1337`** thay vì `φ(1337)`; (b) **dùng nguyên `B mod φ` (không `+ φ`)** khi `a` không nguyên tố cùng nhau với `1337`; (c) **bóc chữ số sai công thức** (nhân `a · d` thay vì `a^d`). Ta chứng minh chúng cho kết quả **khác** đúng ở một số input.

In [27]:
def wrong_mod_1337_exponent(a, b):
    r = 0
    for d in b:
        r = (r * 10 + d) % MOD                    # SAI: số mũ phải theo φ(1337) = 1140, không phải 1337
    return pow(a, r, MOD)


def wrong_euler_no_plus_phi(a, b):
    r = 0
    for d in b:
        r = (r * 10 + d) % PHI
    return pow(a % MOD, r, MOD)                   # SAI khi gcd(a, 1337) > 1 và B >= φ


def wrong_digit_formula(a, b):
    a %= MOD
    result = 1
    for d in b:
        result = pow(result, 10, MOD) * (a * d) % MOD     # SAI: a * d thay vì a ** d
    return result


for fn, a, b in [(wrong_mod_1337_exponent, 2, [1, 3, 3, 7]),
                 (wrong_euler_no_plus_phi, 7, [1, 1, 4, 0]),
                 (wrong_digit_formula, 2, [1, 2, 3])]:
    print(fn.__name__, fn(a, b), "!=", sol.superPow(a, b))
    assert fn(a, b) != sol.superPow(a, b)

wrong_mod_1337_exponent 1 != 319
wrong_euler_no_plus_phi 1 != 574
wrong_digit_formula 559 != 1191


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: `1 <= a <= 2^31 - 1`, `1 <= len(b) <= 2000`, chữ số `0..9`, **chữ số đầu khác `0`**. Ta kiểm tra:

1. **Vét cạn theo lớp dư của `a`**: mọi `a` trong `0..1336` (mỗi lớp dư) với nhiều giá trị `B` quanh các điểm đặc biệt (`1`, `10`, `1139`, `1140`, `1141`, `2280`, ...), cho cả 5 cách (naive chỉ với `B` nhỏ).
2. **Ngẫu nhiên có seed**: `a` ngẫu nhiên tới `2^31 - 1`, `b` dài `1..60`.

In [28]:
special_B = list(range(1, 41)) + [100, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1337, 1338, 2279, 2280, 2281, 9999, 10000, 123456]
fast_keys = ("fastpow", "digits", "euler", "cycle", "leetcode")
for a in range(1, 1338 + 7):                      # mọi lớp dư mod 1337 (kể cả a ≡ 0 qua a = 1337)
    for B in special_B:
        b = [int(ch) for ch in str(B)]
        ref = pow(a, B, MOD)
        for key in fast_keys:
            assert impls[key](a, list(b)) == ref, (key, a, B)
        if B <= 40 and a <= 200:
            assert super_pow_naive(a, list(b)) == ref
print("PASS vét cạn: 1344 giá trị a x", len(special_B), "giá trị B (5 cách + naive khi nhỏ)")

rng = random.Random(372)
for trial in range(2000):
    a = rng.choice([rng.randint(1, 2 ** 31 - 1), rng.randint(1, 5000), rng.choice([7, 191, 1337, 7 * 191 * 3])])
    n = rng.randint(1, 60)
    b = [rng.randint(1, 9)] + [rng.randint(0, 9) for _ in range(n - 1)]
    assert len(b) == n and b[0] != 0
    ref = oracle(a, b)
    for key in fast_keys:
        assert impls[key](a, list(b)) == ref, (key, a, b)
    if digits_to_int(b) <= 3000:
        assert super_pow_naive(a, list(b)) == ref
print("PASS 2000 test ngẫu nhiên (a tới 2^31 - 1, b dài 1..60)")

PASS vét cạn: 1344 giá trị a x 56 giá trị B (5 cách + naive khi nhỏ)
PASS 2000 test ngẫu nhiên (a tới 2^31 - 1, b dài 1..60)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính `a ** B` trực tiếp rồi `% 1337`

In [29]:
a, b = 3, [1, 0, 0, 0, 0]          # B = 10000: vẫn nhỏ
big_number = a ** digits_to_int(b)
digits_count = int(10000 * math.log10(3)) + 1     # số chữ số thập phân ước lượng bằng log10
print("3 ** 10000 có khoảng", digits_count, "chữ số,", big_number.bit_length(), "bit; % 1337 =", big_number % 1337, "| đúng:", Solution().superPow(a, b))
# Với B = 10^2000: số 3 ** B có khoảng 0.477 * 10^2000 chữ số -> không thể tính nổi.

3 ** 10000 có khoảng 4772 chữ số, 15850 bit; % 1337 = 578 | đúng: 578


**Vì sao sai:** `a ** B` sinh ra một số có `~B · log10(a)` chữ số — với `B` tới `10^2000` là **không thể** lưu hay tính. Luôn **lấy `mod` ở mọi phép nhân** (hoặc dùng `pow(a, e, m)`). **Cách sửa:** `result = result * a % 1337` hoặc `pow(x, e, 1337)`.

### Lỗi 2: Rút gọn số mũ theo `1337` thay vì `φ(1337) = 1140`

In [30]:
print("sai:", wrong_mod_1337_exponent(2, [1, 3, 3, 7]), "| đúng:", Solution().superPow(2, [1, 3, 3, 7]))
print("pow(2, 1337, 1337) =", pow(2, 1337, 1337), "vs pow(2, 0, 1337) =", pow(2, 0, 1337))

sai: 1 | đúng: 319
pow(2, 1337, 1337) = 319 vs pow(2, 0, 1337) = 1


**Vì sao sai:** chu kỳ của lũy thừa modulo `m` liên quan tới **`φ(m)`** (hay chính xác là bậc của `a`), **không phải** `m`. Với `B = 1337`, `2^1337 ≢ 2^0`. **Cách sửa:** rút gọn theo `φ(1337) = 1140` (cùng điều kiện ở Lỗi 3) hoặc bóc chữ số (Cách 3).

### Lỗi 3: Dùng định lý Euler khi `gcd(a, 1337) > 1` mà bỏ qua trường hợp biên

In [31]:
print("a = 7, B = 1140")
print("  7^1140 mod 1337 đúng        =", pow(7, 1140, MOD))
print("  7^(1140 mod 1140) = 7^0     =", pow(7, 0, MOD), " (sai)")
print("  wrong_euler_no_plus_phi     =", wrong_euler_no_plus_phi(7, [1, 1, 4, 0]))
print("  Solution                    =", Solution().superPow(7, [1, 1, 4, 0]))
print("a = 3 (gcd = 1): 3^1140 mod 1337 =", pow(3, 1140, MOD), "(đúng bằng 1, định lý Euler)")

a = 7, B = 1140
  7^1140 mod 1337 đúng        = 574
  7^(1140 mod 1140) = 7^0     = 1  (sai)
  wrong_euler_no_plus_phi     = 1
  Solution                    = 574
a = 3 (gcd = 1): 3^1140 mod 1337 = 1 (đúng bằng 1, định lý Euler)


**Vì sao sai:** định lý Euler `a^φ(m) ≡ 1` chỉ đúng khi `gcd(a, m) = 1`. Với `a = 7` (chia hết `7 | 1337`), mọi lũy thừa `7^k` (với `k ≥ 1`) đều chia hết cho `7`, không bao giờ bằng `1` modulo `1337`. **Cách sửa:** dùng dạng mở rộng `a^(B mod φ + φ)` khi `B ≥ φ`, hoặc tránh hẳn Euler và dùng Cách 3/Cách 5.

### Lỗi 4: Nhầm công thức bóc chữ số (`a · d` thay vì `a^d`)

In [32]:
print("sai:", wrong_digit_formula(2, [1, 2, 3]), "| đúng:", Solution().superPow(2, [1, 2, 3]))

sai: 559 | đúng: 1191


**Vì sao sai:** thêm chữ số `d` vào bên phải số mũ nghĩa là **nhân thêm `a^d`** (mũ `d`), chứ không phải nhân `a · d`. **Cách sửa:** `pow(a, d, mod)`.

### Lỗi 5: Đệ quy sâu `2000` tầng (và cắt lát `b[:-1]` tốn thêm)

In [33]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    super_pow_recursive(2, [1] * 2000)             # 2000 tầng đệ quy > giới hạn mặc định
except RecursionError as err:
    print("RecursionError:", str(err)[:60])
print("bản lặp:", Solution().superPow(2, [1] * 2000) == pow(2, int("1" * 2000), MOD))

giới hạn đệ quy mặc định: 1000
RecursionError: maximum recursion depth exceeded
bản lặp: True


**Vì sao sai:** `b` dài `2000` nên bản đệ quy gọi tới `2000` tầng, vượt giới hạn `1000` của Python; mỗi lần cắt `b[:-1]` còn tốn `O(n)` ⇒ `O(n²)`. **Cách sửa:** dùng vòng lặp trái → phải (Cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Lũy thừa với số mũ khổng lồ, kết quả modulo**" ⇒ lập tức nghĩ tới: **lũy thừa nhanh**, **bóc chữ số** `a^(10x+d) = (a^x)^10 · a^d`, hoặc **định lý Euler / Fermat**.
- Mọi phép nhân đi kèm `% mod`. Trong Python, `pow(x, e, m)` đã làm sẵn và rất nhanh.
- Khi số mũ cho dưới dạng **mảng chữ số / chuỗi**, nghĩ ngay tới cách đọc **từng chữ số** (Horner) — hoặc `B mod k` bằng `r = (r * 10 + d) % k`.
- `1337 = 7 × 191`, `φ(1337) = 6 × 190 = 1140`. Luôn kiểm tra `gcd(a, 1337)` trước khi dùng Euler; dạng mở rộng `a^(B mod φ + φ)` đúng với mọi `a` khi `B ≥ φ`.
- Dãy `a^k mod m` luôn **tuần hoàn từ một lúc nào đó**; ta có thể dò chu kỳ bằng `dict`.
- Luôn tạo **oracle** bằng `pow(a, B, m)` để kiểm tra mọi cách giải.
- Trong C++/Java: dùng bóc chữ số (không có số lớn sẵn) và cẩn thận **tràn số** (`long long` khi nhân hai số `< 1337` là đủ).

## 15. Biến thể của bài

1. **Pow(x, n) (50)** — lũy thừa nhanh với số mũ nguyên 32 bit (kể cả âm).
2. **Modular exponentiation với `b` là chuỗi** — cùng ý bóc chữ số, đổi `mod` bất kỳ.
3. **Hai chữ số cuối của `a^B`** (`mod 100`, `φ(100) = 40`).
4. **Tháp lũy thừa `a^(b^c) mod m`** — dùng định lý Euler **đệ quy** (`b^c mod φ(m)` rồi lũy thừa).
5. **Định lý số dư Trung Hoa (CRT)** — tính `mod 7` và `mod 191` riêng rồi ghép lại.
6. **Count Primes / Fermat primality test** — dùng `pow(a, n - 1, n)`.

In [34]:
def super_pow_mod(a, b, m):
    """Cùng ý Cách 3 với mô-đun bất kỳ m."""
    a %= m
    result = 1 % m
    for d in b:
        result = pow(result, 10, m) * pow(a, d, m) % m
    return result


def last_two_digits(a, b):
    return super_pow_mod(a, b, 100)


def super_pow_crt(a, b):
    """CRT: tính a^B mod 7 và mod 191 rồi ghép lại (1337 = 7 * 191)."""
    B = digits_to_int(b)
    r7 = pow(a % 7, B, 7)
    r191 = pow(a % 191, B, 191)
    # x = r7 (mod 7), x = r191 (mod 191)
    inv = pow(7, -1, 191)                             # nghịch đảo của 7 modulo 191
    k = ((r191 - r7) * inv) % 191
    return r7 + 7 * k


def tower_pow(a, b, c, m):
    """a^(b^c) mod m, tính bằng pow() lồng nhau (kiểm chứng), so với dùng định lý Euler cho số mũ."""
    return pow(a, pow(b, c), m)


print(super_pow_mod(2, [1, 0], 1000), last_two_digits(7, [4]), super_pow_crt(2, [1, 0]), tower_pow(2, 3, 4, 1337))
for a in range(1, 200):
    for B in (1, 2, 17, 1140, 1141, 5000):
        b = [int(ch) for ch in str(B)]
        assert super_pow_crt(a, b) == pow(a, B, MOD)
        assert last_two_digits(a, b) == pow(a, B, 100)
print("super_pow_crt và last_two_digits đúng")

24 1 1024 50
super_pow_crt và last_two_digits đúng


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nhân `B` lần | đúng định nghĩa | `O(B)` | `O(1)` | `B` rất nhỏ; làm **oracle phụ** |
| 2. Ghép số + lũy thừa nhanh | bình phương liên tiếp trên `B` | `~O(n²)` | `O(n)` | Python/Java có số lớn |
| 3. Bóc chữ số | `a^(10x+d) = (a^x)^10 · a^d` | `O(n)` | `O(1)` | **nên nộp**, dùng được mọi ngôn ngữ |
| 4. Định lý Euler | `B mod φ(1337)` (+ `φ` nếu cần) | `O(n)` | `O(1)` | hiểu số học; cẩn thận `gcd > 1` |
| 5. Dò chu kỳ | dãy lũy thừa tuần hoàn, `B mod λ` | `O(1337 + n)` | `O(1337)` | không cần định lý; trực quan |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(7 * 8) % 5`, `divmod(123, 10)`, `pow(2, 10, 1000)`, `Solution().superPow(2, [3])`, `Solution().superPow(10, [2])`.

**Bài 2 (tính tay).** Với `a = 2`, `b = [1, 3]` (tức `2^13`): dùng công thức `result = result^10 · a^d` cho từng chữ số rồi lấy `mod 1337`. So với `2^13 = 8192`.

**Bài 3 (điền TODO).** Hoàn thành `super_pow_todo(a, b)` theo Cách 3.

In [35]:
def super_pow_todo(a, b):
    # TODO 1: a %= 1337 và result = 1
    # TODO 2: với mỗi chữ số d trong b: result = pow(result, 10, 1337) * pow(a, d, 1337) % 1337
    # TODO 3: trả về result
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `super_pow_right_to_left(a, b)` bóc chữ số **từ phải sang trái**: giữ `base = a^(10^i)` ứng với vị trí `i` và nhân `base^d` vào kết quả, rồi `base = base^10`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `euler_phi(n)` (đã có ở mục 3.5, hãy tự viết lại bằng phân tích thừa số) và kiểm tra `euler_phi(1337) == 1140`. (b) Viết `last_two_digits(a, b)` cho `a^B mod 100` (`b` là mảng chữ số). (c) Vì sao Cách 3 vẫn đúng nếu đề thay `1337` bằng một mô-đun **bất kỳ** (kể cả không nguyên tố)?

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `56 % 5 = 1`, `(12, 3)`, `2^10 = 1024` nên `1024 % 1000 = 24`; `superPow(2, [3]) = 8`; `10^2 = 100`. Bài 3: chép lại hai dòng chính của Cách 3.
- **Mức 2 (vừa):** Bài 2: chữ số `1`: `1^10 · 2^1 = 2`. Chữ số `3`: `2^10 · 2^3 = 1024 · 8 = 8192`, `8192 mod 1337 = 8192 - 6·1337 = 170`. Bài 4: duyệt `reversed(b)`; nhớ `base = pow(base, 10, 1337)` ở cuối mỗi vòng.
- **Mức 3 (sâu):** Bài 5(a): chia thừa số nguyên tố `p` rồi `result -= result // p`. 5(b): thay `1337` bằng `100`. 5(c): vì `a^(10x+d) = (a^x)^10 · a^d` là **đồng nhất thức của số mũ** nên đúng với mọi mô-đun; và `(u · v) mod m = ((u mod m)(v mod m)) mod m` đúng với mọi `m`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [36]:
# Bài 1
assert (7 * 8) % 5 == 1
assert divmod(123, 10) == (12, 3)
assert pow(2, 10, 1000) == 24
assert Solution().superPow(2, [3]) == 8
assert Solution().superPow(10, [2]) == 100

# Bài 2
result = 1
for d in [1, 3]:
    result = pow(result, 10, MOD) * pow(2, d, MOD) % MOD
assert result == 8192 % MOD == 170


# Bài 3
def super_pow_todo_ans(a, b):
    a %= 1337
    result = 1
    for d in b:
        result = pow(result, 10, 1337) * pow(a, d, 1337) % 1337
    return result


for a, b, expected in examples:
    assert super_pow_todo_ans(a, b) == expected


# Bài 4
def super_pow_right_to_left(a, b):
    base = a % MOD                         # base = a^(10^i) với i = vị trí từ phải
    result = 1
    for d in reversed(b):
        result = result * pow(base, d, MOD) % MOD
        base = pow(base, 10, MOD)
    return result


for a, b, expected in examples:
    assert super_pow_right_to_left(a, b) == expected
rng2 = random.Random(5)
for _ in range(300):
    a = rng2.randint(1, 10 ** 9)
    b = [rng2.randint(1, 9)] + [rng2.randint(0, 9) for _ in range(rng2.randint(0, 30))]
    assert super_pow_right_to_left(a, b) == oracle(a, b)


# Bài 5
def euler_phi2(n):
    result, m, p = n, n, 2
    while p * p <= m:
        if m % p == 0:
            while m % p == 0:
                m //= p
            result -= result // p
        p += 1
    if m > 1:
        result -= result // m
    return result


assert euler_phi2(1337) == 1140
assert [euler_phi2(n) for n in (1, 2, 9, 10, 100)] == [1, 1, 6, 4, 40]


def last_two_digits_ans(a, b):
    a %= 100
    result = 1
    for d in b:
        result = pow(result, 10, 100) * pow(a, d, 100) % 100
    return result


for a in range(1, 150):
    for B in (1, 2, 3, 20, 21, 40, 41, 99):
        assert last_two_digits_ans(a, [int(c) for c in str(B)]) == pow(a, B, 100)
for m in (2, 10, 100, 1000, 1337, 97):
    for a in range(1, 60):
        assert super_pow_mod(a, [1, 2, 3, 4], m) == pow(a, 1234, m)
print("Đáp án bài 1-5 đều đúng")

Đáp án bài 1-5 đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lũy thừa với số mũ là mảng chữ số, kết quả modulo)

```python
class Solution:
    def superPow(self, a: int, b: List[int]) -> int:
        mod = 1337
        a %= mod
        result = 1
        for d in b:
            result = pow(result, 10, mod) * pow(a, d, mod) % mod
        return result
```

**Checklist**

- [ ] Đã `a %= 1337` ngay từ đầu và `% mod` sau mọi phép nhân (hoặc dùng `pow(x, e, mod)`)?
- [ ] `result` bắt đầu là `1` (tức `a^0`), không phải `a` hay `0`?
- [ ] Công thức đúng: `result = result^10 · a^d` (mũ `10` cho `result`, mũ `d` cho `a`)?
- [ ] Đọc `b` từ **trái sang phải** (chữ số cao trước)?
- [ ] Không dùng `a ** B`; không đệ quy sâu `2000` tầng?
- [ ] Nếu dùng Euler: `φ(1337) = 1140`, và xử lý `gcd(a, 1337) > 1` bằng `+ φ` (hoặc tránh Euler)?
- [ ] Đã thử `a = 1`, `a = 1337`, `b = [1]`, `b = [1,1,4,0]` và `b` dài `2000` chữ số?